<a href="https://colab.research.google.com/github/Elshekh00/molecular-dynamics-protein-ligand/blob/main/md_simulation_protein_ligand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hello there!**
# **Molecular Dynamics Simulation of Protein–Ligand Complexes**

**Author:** Ahmed Elshekh  
**Platform:** Google Colab  
**Simulation Engine:** OpenMM  
**Force Field:** AMBER  

---

 **This notebook is NOT a standard protocol for MD simulations!** It is just simple MD pipeline illustrating each step of a simulation protocol.

---
### **Overview**
This notebook provides a reproducible, cloud-based workflow for conducting molecular dynamics (MD) simulations of protein–ligand complexes. The protocol encompasses all essential stages of an MD study, including system preparation, energy minimization, equilibration under controlled thermodynamic ensembles, production simulations, and post-simulation trajectory analysis.

### **Scope and Rationale**
Molecular dynamics simulations offer atomistic insights into the structural stability, conformational dynamics, and interaction patterns of biomolecular systems. By leveraging a cloud-computing environment, this workflow enables efficient execution of MD simulations without the need for dedicated high-performance computing infrastructure, thereby enhancing accessibility while preserving methodological rigor.

### **Reproducibility**
All simulation steps are implemented using standardized parameters and open-source tools to ensure transparency and reproducibility. The workflow is structured to allow straightforward adaptation to different protein–ligand systems with minimal modification.

## **Introduction**

Molecular dynamics (MD) simulations are computational techniques used to study the time-dependent behavior of molecular systems at the atomic level. These simulations rely on two essential components:

1. Atomic coordinates of the system (e.g., protein and ligand structures).
2. A force field that defines the interaction energies between atoms.

In this workflow, a protein–ligand complex is simulated using the AMBER force field within the OpenMM simulation engine. Ligand parameters are generated using the General AMBER Force Field (GAFF), which supports a wide range of organic molecules.

The system preparation involves building the simulation box, solvation, and ion addition, followed by energy minimization and equilibration steps prior to the production MD simulation.

## **Software and Dependencies**

The following software packages and libraries are required to perform the molecular dynamics simulations:

- [OpenMM](https://openmm.org/)
- [AmberTools](https://ambermd.org/AmberTools.php)
- [PyTraj](https://amber-md.github.io/pytraj/latest/index.html)
- [py3Dmol](https://pypi.org/project/py3Dmol/)
- [ProLIF](https://github.com/chemosim-lab/ProLIF)
- [NumPy](https://numpy.org/)
- [Matplotlib](https://matplotlib.org/)

All required dependencies are installed automatically within the Google Colab environment as part of the workflow.

## **1. Environment Setup: Conda Installation**
 It will restart the session, don't worry.

In [ ]:
#@title **Install Miniforge (Conda)**
# !pip install -q condacolab
# import condacolab
# condacolab.install()
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")
)

## **2. Package Installation and Library Setup**

All required scientific libraries and simulation dependencies are installed in this step using both Conda (via mamba) and pip. Version-specific installations are applied where necessary to ensure compatibility and reproducibility of the molecular dynamics workflow.

This step may take several minutes depending on the runtime environment.

In [ ]:
#@title **Install dependencies**
# install dependencies
import subprocess
import sys
subprocess.run("pip install numpy==2.0.0", shell=True)
import numpy as np
print(np.__version__)
subprocess.run("mamba install -c conda-forge ambertools -y", shell=True)
import pytraj as pt
subprocess.run("pip -q install py3Dmol", shell=True)
subprocess.run("pip install git+https://github.com/pablo-arantes/biopandas", shell=True)
subprocess.run("mamba install openmm pdbfixer -y", shell=True)
subprocess.run("pip install rdkit==2025.3.1", shell=True)
subprocess.run("pip install prolif==1.1.0", shell=True)
subprocess.run("mamba install -c conda-forge openbabel -y", shell=True)
subprocess.run("mamba install -c conda-forge parmed -y", shell=True)
subprocess.run("mamba install -c conda-forge mdtraj -y", shell=True)
subprocess.run("pip install MDAnalysis==2.8.0", shell=True)

#load dependencies
from openmm import app, unit
from openmm.app import HBonds, NoCutoff, PDBFile
import parmed as pmd
from biopandas.pdb import PandasPdb
import openmm as mm
from openmm import *
from openmm.app import *
from openmm.unit import *
import os
import urllib.request
import numpy as np
import MDAnalysis as mda
import py3Dmol
import pytraj as pt
import platform
import scipy.cluster.hierarchy
from scipy.spatial.distance import squareform
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import griddata
import seaborn as sb
from statistics import mean, stdev
from pytraj import matrix
from matplotlib import colors
from IPython.display import set_matplotlib_formats

## **3. Data Storage and File Management**

Due to the temporary nature of the Google Colab runtime environment, local data storage is not persistent across sessions. Therefore, external storage is required to retain simulation inputs and outputs.

In this workflow, Google Drive is utilized as a persistent storage solution to enable reliable access to input files and preservation of simulation results.

Users should prepare a dedicated directory within their Google Drive containing all required input files (e.g., protein and ligand structures) prior to execution of the simulation workflow. The path to this directory will be specified and used in subsequent steps.

In [ ]:
#@title ### **Import Google Drive**
from google.colab import drive

drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

In [ ]:
#@title **Check if you correctly allocated GPU nodes**

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Select the Runtime > "Change runtime type" menu to enable a GPU accelerator, ')
  print('and then re-execute this cell.')
else:
  print(gpu_info)

---
## **4. Input File Preparation and System Definition**

At this stage, all required software dependencies should be installed, and the necessary input files should be available within the designated Google Drive directory.

The simulation requires the following input files:
- A protein structure file in PDB format
- A ligand structure file in PDB format

The protein and ligand coordinates must be structurally consistent, with the ligand correctly positioned within the binding site of the receptor. This is critical for ensuring meaningful simulation results.

The file paths corresponding to the input directory should be specified accurately. Incorrect file paths or improperly formatted filenames may lead to execution errors.

**Note:** It is recommended to avoid using spaces in file and directory names to ensure compatibility with command-line operations and scripting environments.

In [ ]:
#@title **Please, Define input files below**:
#@markdown **Important:** Protonation of your ligand is required for the accurate parameterization of the molecule..
# %%capture
import rdkit
import mdtraj as md
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem import rdMolTransforms
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem import rdDepictor
from rdkit.Chem import rdForceFieldHelpers
from IPython.display import SVG
import ipywidgets as widgets
import rdkit
from rdkit.Chem.Draw import IPythonConsole
AllChem.SetPreferCoordGen(True)
from IPython.display import Image
from pdbfixer import PDBFixer
from openbabel import pybel
import os
import subprocess
import warnings

# Suppress UserWarnings
warnings.filterwarnings("ignore", category=UserWarning)


Protein_PDB_file_name = 'protein.pdb' #@param {type:"string"}
remove_waters = "yes" #@param ["yes", "no" ]
if remove_waters == "yes":
  no_waters = "nowat"
else:
  no_waters = ''

Ligand_PDB_file_name = 'ligand.pdb'  #@param {type:"string"}

Add_ligand_hydrogens = "Yes" #@param ["Yes", "No"]
Charge = 0 #@param {type:"slider", min:-10, max:10, step:1}


ligand_name = Ligand_PDB_file_name
Google_Drive_Path = '/content/drive/MyDrive/' #@param {type:"string"}
workDir = Google_Drive_Path
initial_pdb = os.path.join(workDir, str(Protein_PDB_file_name))
prepareforleap = os.path.join(workDir, "prepareforleap.in")
ligand_pdb = os.path.join(workDir, str(ligand_name))
ligand_pdb2 = os.path.join(workDir, "ligand_H.pdb")
starting = os.path.join(workDir, "starting1.pdb")
starting2 = os.path.join(workDir, "starting2.pdb")
starting_end = os.path.join(workDir, "starting_end.pdb")


def mol_with_atom_index(mol):
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(atom.GetIdx())
    return mol

def remove_lines(filename):
    with open(filename, 'r') as file:
        ter_count = 0
        for line in file:
            if line.startswith('TER'):
                ter_count += 1
                if ter_count >= 1:
                    yield line
                    for i in range(3):
                        line = next(file, None)
                        if line is not None and line.startswith('ATOM') and line.split()[2] in ['P', 'OP1', 'OP2']:
                            continue
                        else:
                            yield line
                else:
                    yield line
            else:
                yield line

# if Add_ligand_hydrogens == "Yes":
#   mol= [m for m in pybel.readfile(filename=ligand_pdb, format='pdb')][0]
#   out=pybel.Outputfile(filename="temp.mol",format='mol',overwrite=True)
#   out.write(mol)
#   out.close()

#   mol = Chem.MolFromMolFile('temp.mol', removeHs=True)
#   hmol = Chem.AddHs(mol)
#   mp = AllChem.MMFFGetMoleculeProperties(hmol)
#   ff = AllChem.MMFFGetMoleculeForceField(hmol, mp)
#   for a in hmol.GetAtoms():
#     if (a.GetAtomicNum() > 1):
#       ff.MMFFAddPositionConstraint(a.GetIdx(), 0, 1.e4)
#   ff.Minimize(maxIts=1000)
#   charge_mol = Chem.rdPartialCharges.ComputeGasteigerCharges(hmol)
#   charge = Chem.GetFormalCharge(hmol)
#   print("Charge = " + str(charge))
#   # AllChem.MolToMolFile(hmol, (os.path.join(workDir, f"start_min.mol")))
#   AllChem.MolToPDBFile(hmol, ligand_pdb2)
#   mol_end = mol_with_atom_index(hmol)
#   IPythonConsole.drawMol3D(hmol)
# else:
#   mol= [m for m in pybel.readfile(filename=ligand_pdb, format='pdb')][0]
#   out=pybel.Outputfile(filename="temp.mol",format='mol',overwrite=True)
#   out.write(mol)
#   out.close()

#   hmol = Chem.MolFromMolFile('temp.mol', removeHs=False)
#   mp = AllChem.MMFFGetMoleculeProperties(hmol)
#   ff = AllChem.MMFFGetMoleculeForceField(hmol, mp)
#   for a in hmol.GetAtoms():
#     if (a.GetAtomicNum() > 1):
#       ff.MMFFAddPositionConstraint(a.GetIdx(), 0, 1.e4)
#   ff.Minimize(maxIts=1000)
#   charge_mol = Chem.rdPartialCharges.ComputeGasteigerCharges(hmol)
#   charge = Chem.GetFormalCharge(hmol)
#   print("Charge = " + str(charge))
#   # AllChem.MolToMolFile(hmol, (os.path.join(workDir, f"start_min.mol")))
#   AllChem.MolToPDBFile(hmol, ligand_pdb2)
#   mol_end = mol_with_atom_index(hmol)
#   IPythonConsole.drawMol3D(hmol)



#Add hydrogens in the ligand
if Add_ligand_hydrogens == "Yes":
  fixer = PDBFixer(filename=ligand_pdb)
  PDBFile.writeFile(fixer.topology, fixer.positions, open("temp.pdb", 'w'))

  ppdb = PandasPdb().read_pdb("temp.pdb")
  ppdb.df['ATOM'] = ppdb.df['ATOM']
  ppdb.df['HETATM']= ppdb.df['HETATM'][ppdb.df['HETATM']['element_symbol'] != 'H']
  ppdb.to_pdb(path="temp.pdb", records=['ATOM', 'HETATM'], gz=False, append_newline=True)

  mol= [m for m in pybel.readfile(filename="temp.pdb", format='pdb')][0]
  mol.calccharges
  mol.addh()
  out=pybel.Outputfile(filename="temp2.pdb",format='pdb',overwrite=True)
  out.write(mol)
  out.close()

  md.load("temp2.pdb").save("temp2.pdb")

  halogens = ['Cl', 'F', 'Br', 'I']
  atom_id = []
  H_id = []
  with open("temp2.pdb") as f:
      for line in f:
        data = line.split()
        if data[0] == "ATOM":
          if data[2] in halogens:
            atom_id.append(data[1])
        if data[0] == "CONECT":
          if data[1] in atom_id:
            if len(data) > 3:
              H_id.append(data[3])
              H_id.append(data[4])
              H_id.append(data[5])

  # with open(ligand_pdb2, 'w') as h:
  #   with open("temp2.pdb") as f:
  #     for line in f:
  #       data = line.split()
  #       if data[0] == "ATOM":
  #         if data[1] not in H_id:
  #           print(line, file=h)
  #       elif data[0] == "CONECT":
  #           if data[1] not in atom_id:
  #             print(line, file=h)
  #       else:
  #         print(line, file=h)

  with open(ligand_pdb2, 'w') as h:
    with open("temp2.pdb") as f:
        for line in f:
            if line.strip():  # Check if line is not empty or just whitespace
                data = line.split()
                if len(data) > 0 and data[0] not in ["TER", "ENDMDL"]:  # Exclude lines starting with TER or ENDMDL
                    if data[0] == "ATOM":
                        if data[1] not in H_id:
                            print(line, end='', file=h)  # Avoid adding extra newline
                    elif data[0] == "CONECT":
                        if data[1] not in atom_id:
                            print(line, end='', file=h)
                    else:
                        print(line, end='', file=h)

  mol= [m for m in pybel.readfile(filename=ligand_pdb2, format='pdb')][0]
  out=pybel.Outputfile(filename="temp.mol",format='mol',overwrite=True)
  out.write(mol)
  out.close()
  hmol = Chem.MolFromMolFile('temp.mol', removeHs=False)
  # charge_mol = Chem.rdPartialCharges.ComputeGasteigerCharges(hmol)
  charge = Charge
  print("Charge = " + str(charge))
  mol_end = mol_with_atom_index(hmol)
  IPythonConsole.drawMol3D(hmol)

else:
  ppdb = PandasPdb().read_pdb(ligand_pdb)
  ppdb.df['ATOM'] = ppdb.df['ATOM']
  ppdb.to_pdb(path="temp.pdb", records=['ATOM', 'HETATM'], gz=False, append_newline=True)
  mol= [m for m in pybel.readfile(filename="temp.pdb", format='pdb')][0]
  mol.calccharges
  out=pybel.Outputfile(filename="temp2.pdb",format='pdb',overwrite=True)
  out.write(mol)
  out.close()

  md.load("temp2.pdb").save("temp2.pdb")

  with open(ligand_pdb2, 'w') as h:
    with open("temp2.pdb") as f:
        for line in f:
            if line.strip() and not line.startswith(("TER", "ENDMDL")):
                print(line, end='', file=h)

  mol= [m for m in pybel.readfile(filename=ligand_pdb2, format='pdb')][0]
  out=pybel.Outputfile(filename="temp.mol",format='mol',overwrite=True)
  out.write(mol)
  out.close()
  hmol = Chem.MolFromMolFile('temp.mol', removeHs=False)
  # charge_mol = Chem.rdPartialCharges.ComputeGasteigerCharges(hmol)
  charge = Charge
  print("Charge = " + str(charge))
  mol_end = mol_with_atom_index(hmol)
  IPythonConsole.drawMol3D(hmol)


#Fix protein
f = open(prepareforleap, "w")
f.write("""parm """ + str(initial_pdb) + "\n"
"""loadcrd """ + str(initial_pdb) + """ name edited""" + "\n"
"""prepareforleap crdset edited name from-prepareforleap \ """ + "\n"
"""pdbout """ + str(starting_end) + " " + str(no_waters) + """ noh""" + "\n"
"""go """)
f.close()

prepareforleap_command = "cpptraj -i " + str(prepareforleap)
original_stdout = sys.stdout # Save a reference to the original standard output
with open('prepareforleap.sh', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    print(prepareforleap_command)
    sys.stdout = original_stdout # Reset the standard output to its original value

subprocess.run(["chmod 700 prepareforleap.sh"], shell=True)
subprocess.run(["./prepareforleap.sh"], shell=True,)


pdb4amber_cmd = "pdb4amber -i " + str(starting) + " -o " + str(starting_end) + " -a"
original_stdout = sys.stdout # Save a reference to the original standard output

with open('pdb4amber.sh', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    print(pdb4amber_cmd)
    sys.stdout = original_stdout # Reset the standard output to its original value

subprocess.run(["chmod 700 pdb4amber.sh"], shell=True)
subprocess.run(["./pdb4amber.sh"], shell=True,)

# with open(starting_end, 'w') as out_file:
#     for line in remove_lines(starting2):
#         out_file.write(line)

protein_check = os.path.exists(starting_end)
ligand_check = os.path.exists(ligand_pdb2)

if protein_check == True and ligand_check == True:
  print("Successfully generated protein and ligand files! :-)")
else:
  print("ERROR: Check your inputs! ")

In [ ]:
from rdkit import Chem
import parmed

#@title **Topology Generation and System Assembly:**

#@markdown **Define parameters for topology generation and system setup:**

Force_field = "ff19SB" #@param ["ff19SB", "ff14SB"]
if Force_field == "ff19SB":
  ff = "leaprc.protein.ff19SB"
else:
  ff = "leaprc.protein.ff14SB"

Water_type = "TIP3P" #@param ["TIP3P", "OPC"]
if Water_type == "TIP3P":
  water = "leaprc.water.tip3p"
  water_box = "TIP3PBOX"
else:
  water = "leaprc.water.opc"
  water_box = "OPCBOX"

#@markdown Size Box (Angstrons):

Size_box = 12 #@param {type:"slider", min:10, max:20, step:1}
size_box = Size_box

#@markdown **ATTENTION**: Enter the concentration in molar units, and AMBER tleap will automatically neutralize your system:

Ions = "NaCl" #@param ["NaCl", "KCl" ]

Concentration = "0.15" #@param {type:"string"}

#@markdown **Parameters to generate the ligand topology:**

Ligand_Force_field = "GAFF2" #@param ["GAFF2"]

# Ligand_isomer = "1" #@param {type:"string", min:1, max:10, step:100}
# if chiralCentres == []:
#   isomer_end = isomer1
# else:
#   isomer_end = globals()[f"isomer{Ligand_isomer}"]

Ligand_net_charges = charge

#@markdown ---
tleap = os.path.join(workDir, "tleap.in")
top_nw = os.path.join(workDir, "SYS_nw.prmtop")
crd_nw = os.path.join(workDir, "SYS_nw.crd")
pdb_nw = os.path.join(workDir, "SYS_nw.pdb")
top = os.path.join(workDir, "SYS_gaff2.prmtop")
crd = os.path.join(workDir, "SYS_gaff2.crd")
pdb = os.path.join(workDir, "SYS.pdb")
ligand_noh = os.path.join(workDir, "ligand_noh.pdb")
ligand_h = os.path.join(workDir, "ligand_h.pdb")
ligand_mol2 = os.path.join(workDir, "ligand.mol2")
ligand_frcmod = os.path.join(workDir, "ligand.frcmod")
lig_new = os.path.join(workDir, "ligand_gaff.pdb")
protein_ligand = os.path.join(workDir, "protein_ligand.pdb")
lib = os.path.join(workDir, "lig.lib")

#gaff_command1 = "pdb4amber -i " + str(ligand_pdb2) + " -o " + str(ligand_h)
gaff_command1 = "pdb4amber -i " + str(ligand_pdb2) + " -o " + str(ligand_h)
gaff_command3 = "antechamber -i " + str(ligand_h) + " -fi pdb -o " + str(ligand_mol2) + " -fo mol2 -c bcc -nc " + str(Ligand_net_charges) + " -rn LIG -at gaff2"
gaff_command4 = "parmchk2 -i " + str(ligand_mol2) + " -f mol2 -o " + str(ligand_frcmod) + " -s gaff2"

original_stdout = sys.stdout # Save a reference to the original standard output

with open('gaff.sh', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    print(gaff_command1)
    print(gaff_command3)
    print(gaff_command4)
    sys.stdout = original_stdout # Reset the standard output to its original value

!chmod 700 gaff.sh 2>&1 1>/dev/null
!bash gaff.sh >/dev/null 2>&1

f = open(tleap, "w")
f.write("""source """ + str(ff) + "\n"
"""source leaprc.gaff2
LIG = loadmol2 """ + str(ligand_mol2) + "\n"
"""loadamberparams """ + str(ligand_frcmod) + "\n"
"""saveoff LIG """ + str(lib) + "\n"
"""savepdb LIG """ + str(lig_new) + "\n"
"""quit""")
f.close()

tleap_command = "tleap -f " + str(tleap)
cat_command = "cat " + str(starting_end) + " " + str(lig_new) + str(" > ") + str(protein_ligand)

original_stdout = sys.stdout # Save a reference to the original standard output

with open('run_tleap.sh', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    print(tleap_command)
    print(cat_command)
    sys.stdout = original_stdout # Reset the standard output to its original value

!chmod 700 run_tleap.sh 2>&1 1>/dev/null
!bash run_tleap.sh 2>&1 1>/dev/null

ppdb = PandasPdb().read_pdb(protein_ligand)
ppdb.df['ATOM'] = ppdb.df['ATOM']
ppdb.df['OTHERS'] = [ppdb.df['OTHERS'] != 'OTHERS']
ppdb.to_pdb(path=protein_ligand, records=['ATOM', 'HETATM'], gz=False, append_newline=True)

f = open(tleap, "w")
f.write("""source """ + str(ff) + "\n"
"""source leaprc.DNA.OL15
source leaprc.RNA.OL3
source leaprc.GLYCAM_06j-1
source leaprc.gaff2
source """  + str(water) + "\n"
"""loadamberparams """ + str(ligand_frcmod) + "\n"
"""loadoff """ + str(lib) + "\n"
"""SYS = loadpdb """ + str(protein_ligand) + "\n"
"""alignaxes SYS
savepdb SYS """ + str(pdb_nw) + "\n"
"""saveamberparm SYS """ + str(top_nw) + " " + str(crd_nw) + "\n"
"""solvatebox SYS """ + str(water_box) + " " + str(size_box) +  """ 0.7
saveamberparm SYS """ + str(top) + " " + str(crd) + "\n"
"""savepdb SYS """ + str(pdb) + "\n"
"""quit""")
f.close()

tleap_command = "tleap -f " + str(tleap)

original_stdout = sys.stdout # Save a reference to the original standard output

with open('run_tleap.sh', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    print(tleap_command)
    sys.stdout = original_stdout # Reset the standard output to its original value

SYS = os.path.join(workDir, "SYS*")
rm_sys = "rm " + SYS

original_stdout = sys.stdout # Save a reference to the original standard output

with open('rm_sys.sh', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    print(rm_sys)
    sys.stdout = original_stdout # Reset the standard output to its original value

!chmod 700 rm_sys.sh 2>&1 1>/dev/null
!bash rm_sys.sh 2> /dev/null

!chmod 700 run_tleap.sh 2>&1 1>/dev/null
!bash run_tleap.sh 2>&1 1>/dev/null


!grep "Volume:" leap.log > temp.txt
with open("temp.txt", 'r') as f:
  for line in f:
        vol = float(line.split()[1])

vol_lit  = vol * pow(10, -27)
atom_lit = 9.03 * pow(10, 22)
conc = float(Concentration)
num_ion = int(vol_lit * (conc/0.15) * atom_lit)

if Ions == "NaCl":
  pos_neut = "Na+ 0"
  pos_num = "Na+ " + str(num_ion)
  Cl_num = num_ion
else:
  pos_neut = "K+ 0"
  pos_num = "K+ " + str(num_ion)
  Cl_num = num_ion

f = open(tleap, "w")
f.write("""source """ + str(ff) + "\n"
"""source leaprc.DNA.OL15
source leaprc.RNA.OL3
source leaprc.GLYCAM_06j-1
source leaprc.gaff2
source """  + str(water) + "\n"
"""loadamberparams """ + str(ligand_frcmod) + "\n"
"""loadoff """ + str(lib) + "\n"
"""SYS = loadpdb """ + str(protein_ligand) + "\n"
"""alignaxes SYS
check SYS
charge SYS
addions SYS """ + str(pos_neut) + "\n"
"""addions SYS Cl- 0
check SYS
charge SYS
savepdb SYS """ + str(pdb_nw) + "\n"
"""saveamberparm SYS """ + str(top_nw) + " " + str(crd_nw) + "\n"
"""solvatebox SYS """ + str(water_box) + " " + str(size_box) +  """ 0.7 """ + "\n"
"""addIonsRand SYS """ + str(pos_num) + """ Cl- """ + str(Cl_num) + "\n"
"""saveamberparm SYS """ + str(top) + " " + str(crd) + "\n"
"""savepdb SYS """ + str(pdb) + "\n"
"""quit""")
f.close()


!chmod 700 run_tleap.sh 2>&1 1>/dev/null
!bash run_tleap.sh 2>&1 1>/dev/null

# if Ligand_Force_field == "OpenFF 2.0.0 (Sage)":
#   mol = Chem.MolFromPDBFile(lig_new, removeHs=False)
#   Chem.MolToPDBFile(mol, os.path.join(workDir, "ligand_openFF.pdb"))

#   in_prmtop = top
#   in_crd = crd
#   orig_structure = parmed.amber.AmberParm(in_prmtop, in_crd)
#   pieces = orig_structure.split()
#   for piece in pieces:
#       print(f"There are {len(piece[1])} instance(s) of {piece[0]}")

#   from openmm.app import PDBFile
#   from openff.toolkit.topology import Molecule, Topology
#   from openff.toolkit.tests.utils import get_data_file_path

#   # rdmol = Chem.MolFromMolFile(os.path.join(workDir, "ligand_openFF.sdf"))
#   # ligand_off_molecule = Molecule.from_rdkit(rdmol, hydrogens_are_explicit=True)

#   ligand_off_molecule = Molecule.from_smiles(isomer_end)
#   ligand_pdbfile = PDBFile(os.path.join(workDir, "ligand_openFF.pdb"))
#   ligand_off_topology = Topology.from_openmm(
#       ligand_pdbfile.topology,
#       unique_molecules=[ligand_off_molecule],)

#   force_field = ForceField("openff_unconstrained-2.0.0.offxml")

#   ligand_system = force_field.create_openmm_system(ligand_off_topology)
#   new_ligand_structure = parmed.openmm.load_topology(
#       ligand_off_topology.to_openmm(),
#       ligand_system,
#       xyz=pieces[1][0].positions,)

#   new_ligand_structure.save(os.path.join(workDir, "ligand.prmtop"), overwrite=True)
#   new_ligand_structure.save(os.path.join(workDir, "ligand.inpcrd"), overwrite=True)

#   # Check how many atoms and which order elements are in the new ligand
#   n_atoms_new = len(new_ligand_structure.atoms)
#   elements_new = [atom.element for atom in new_ligand_structure.atoms]

#   # Check how many atoms and which order elements are in the old ligand
#   old_ligand_structure, n_copies = pieces[1]
#   n_atoms_old = len(old_ligand_structure.atoms)
#   elements_old = [atom.element for atom in old_ligand_structure.atoms]

#   print(
#       f"There are {n_atoms_old} in the old ligand structure and {n_atoms_new} atoms "
#       f"in the new ligand structure")

#   # Print out error message if number of atoms doesn't match
#   if n_atoms_new != n_atoms_old:
#       print(
#           "Error: Number of atoms in input ligand doesn't match number extracted "
#           "from prmtop file.")

#   if elements_new != elements_old:
#       print(
#           "Error: Elements in input ligand don't match elements in the ligand "
#           "from the prmtop file.")
#       print(f"Old elements: {elements_old}")
#       print(f"New elements: {elements_new}")

#   # Create a new, empty system
#   complex_structure = parmed.Structure()

#   # Add the protein. Convert explicitly to an AmberParm object to ensure that 1-4 scaling factors are preserved.
#   complex_structure += parmed.amber.AmberParm.from_structure(pieces[0][0])

#   # Add the ligand
#   complex_structure += parmed.amber.AmberParm.from_structure(new_ligand_structure)

#   # Add ions and Waters
#   ppdb = PandasPdb().read_pdb(pdb)
#   Cl = [ppdb.df['ATOM']['atom_name'] == 'Cl-']
#   Na = [ppdb.df['ATOM']['atom_name'] == 'Na+']
#   K = [ppdb.df['ATOM']['atom_name'] == 'K+']
#   Cl = np.array(Cl)
#   Na = np.array(Na)
#   K = np.array(K)

#   if True in Cl and True in Na:
#     just_ion1_structure = parmed.Structure()
#     just_ion1_structure += pieces[2][0]
#     just_ion1_structure *= len(pieces[2][1])

#     just_ion2_structure = parmed.Structure()
#     just_ion2_structure += pieces[3][0]
#     just_ion2_structure *= len(pieces[3][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_ion1_structure)
#     complex_structure += parmed.amber.AmberParm.from_structure(just_ion2_structure)

#     just_water_structure = parmed.Structure()
#     just_water_structure += pieces[4][0]
#     just_water_structure *= len(pieces[4][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_water_structure)
#   elif True in Cl and True in K:
#     just_ion1_structure = parmed.Structure()
#     just_ion1_structure += pieces[2][0]
#     just_ion1_structure *= len(pieces[2][1])

#     just_ion2_structure = parmed.Structure()
#     just_ion2_structure += pieces[3][0]
#     just_ion2_structure *= len(pieces[3][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_ion1_structure)
#     complex_structure += parmed.amber.AmberParm.from_structure(just_ion2_structure)

#     just_water_structure = parmed.Structure()
#     just_water_structure += pieces[4][0]
#     just_water_structure *= len(pieces[4][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_water_structure)
#   elif True in Cl:
#     just_ion1_structure = parmed.Structure()
#     just_ion1_structure += pieces[2][0]
#     just_ion1_structure *= len(pieces[2][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_ion1_structure)

#     just_water_structure = parmed.Structure()
#     just_water_structure += pieces[3][0]
#     just_water_structure *= len(pieces[3][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_water_structure)
#   elif True in Na:
#     just_ion1_structure = parmed.Structure()
#     just_ion1_structure += pieces[2][0]
#     just_ion1_structure *= len(pieces[2][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_ion1_structure)

#     just_water_structure = parmed.Structure()
#     just_water_structure += pieces[3][0]
#     just_water_structure *= len(pieces[3][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_water_structure)
#   elif True in K:
#     just_ion1_structure = parmed.Structure()
#     just_ion1_structure += pieces[2][0]
#     just_ion1_structure *= len(pieces[2][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_ion1_structure)

#     just_water_structure = parmed.Structure()
#     just_water_structure += pieces[3][0]
#     just_water_structure *= len(pieces[3][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_water_structure)
#   else:
#     just_water_structure = parmed.Structure()
#     just_water_structure += pieces[2][0]
#     just_water_structure *= len(pieces[2][1])

#     complex_structure += parmed.amber.AmberParm.from_structure(just_water_structure)

#   # Copy over the original coordinates and box vectors
#   complex_structure.coordinates = orig_structure.coordinates
#   complex_structure.box_vectors = orig_structure.box_vectors

#   # Export the Structure to AMBER files
#   top = os.path.join(workDir, "SYS_openff.prmtop")
#   crd = os.path.join(workDir, "SYS_openff.inpcrd")
#   complex_structure.save(top, overwrite=True)
#   complex_structure.save(crd, overwrite=True)
#   top_openff = os.path.exists(top)
#   crd_openff = os.path.exists(crd)
#   if top_openff == True and crd_openff == True:
#     print("Successfully generated topology! :-)")
#   else:
#     print("ERROR: Check your inputs! ")

pdb_amber = os.path.exists(pdb)
top_amber = os.path.exists(top)
crd_amber = os.path.exists(crd)

if pdb_amber == True and top_amber == True and crd_amber == True:
  print("Successfully generated topology! :-)")
else:
  print("ERROR: Check your inputs! ")
!rm *.sh  ANTECHAMBER* ATOMTYPE* temp.txt >/dev/null 2>&1

## **Visualization of the Simulation System:**

In [ ]:
#@title **Visualize the prepared protein–ligand system**
#@markdown This step allows visual inspection of potential structural artifacts prior to simulation.
import warnings
warnings.filterwarnings('ignore')
import py3Dmol

color = "gray" #@param ["gray", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}
show_ligand = True #@param {type:"boolean"}
show_box = True #@param {type:"boolean"}
box_opacity = 0.6 #@param {type:"slider", min:0, max:1, step:0.1}


def show_pdb(show_sidechains=False, show_mainchains=False, show_ligand = False, show_box = False, color="rainbow"):
  view = py3Dmol.view(width=800, height=600)
  view.addModel(open(pdb,'r').read(),'pdb')

  if color == "gray":
    view.setStyle({'cartoon':{}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})

  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                        {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

  if show_box:
    view.addSurface(py3Dmol.SAS, {'opacity': box_opacity, 'color':'white'})

  if show_ligand:
    HP = ['LIG']
    view.addStyle({'and':[{'resn':HP}]},
                       {'stick':{'colorscheme':'greenCarbon','radius':0.3}})
    view.setViewStyle({'style':'outline','color':'black','width':0.1})

  view.zoomTo()
  return view


show_pdb(show_sidechains, show_mainchains, show_ligand, show_box, color).show()

## **5. Protein-Ligand Interaction Network Analysis**

The resulting interaction map can be used to identify key residues involved in ligand binding and to assess the stability of the binding mode prior to simulation.

In [ ]:
#@title **Generate protein–ligand interaction network using LigPlot**
#@markdown The interaction network provides a detailed representation of non-covalent contacts between the ligand and surrounding residues, enabling identification of key stabilizing interactions within the binding pocket.

import MDAnalysis as mda
import prolif as plf
import numpy as np
import os
from prolif.plotting.network import LigNetwork

# load topology
u = mda.Universe(os.path.join(workDir, "SYS_gaff2.prmtop"), pdb)
lig = u.select_atoms("resname LIG")
prot = u.select_atoms("protein")

# create RDKit-like molecules for visualisation
lmol = plf.Molecule.from_mda(lig)
pmol = plf.Molecule.from_mda(prot)

fp = plf.Fingerprint()
fp.run(u.trajectory[::10], lig, prot)
df = fp.to_dataframe(return_atoms=True)

net = LigNetwork.from_ifp(df, lmol,
                          # replace with `kind="frame", frame=0` for the other depiction
                          kind="frame", frame=0,
                          rotation=270)
net.save(os.path.join(workDir, "initial.html"))
net.display()

---
## **6. System Equilibration**

The equilibration step stabilizes the system under controlled temperature and pressure conditions prior to production MD.

During this phase, position restraints are applied to protein heavy atoms while allowing solvent molecules to reorganize around the solute.

Key parameters include temperature, pressure, simulation time, restraint force constants, and trajectory output frequency.

In [ ]:
#@title ### **Define parameters for MD equilibration protocol:**

# remove whitespaces
Jobname = 'prot_lig_equil' #@param {type:"string"}

# Ligand_Force_field = "GAFF2" #@param ["GAFF2", "OpenFF 2.0.0 (Sage)"]

# if Ligand_Force_field == "OpenFF 2.0.0 (Sage)":
#   top = os.path.join(workDir, "SYS_openff.prmtop")
#   crd = os.path.join(workDir, "SYS_openff.inpcrd")
#   pdb = os.path.join(workDir, "SYS.pdb")
# else:
top = os.path.join(workDir, "SYS_gaff2.prmtop")
crd = os.path.join(workDir, "SYS_gaff2.crd")
pdb = os.path.join(workDir, "SYS.pdb")


Minimization_steps = "20000" #@param ["1000", "5000", "10000", "20000", "50000", "100000"]

#@markdown ---
Time (ns)= "5" #@param {type:"string"}
stride_time_eq = Time
Integration_timestep (fs)= "2" #@param ["0.5", "1", "2", "3", "4"]
dt_eq = Integration_timestep

#@markdown ---
Temperature (K)= 298 #@param {type:"string"}
temperature_eq = Temperature
Pressure (bar)= 1 #@param {type:"string"}
pressure_eq = Pressure

#@markdown ---
Force_constant (KJ/mol)= 700 #@param {type:"slider", min:0, max:2000, step:100}

#@markdown ---

Write_the_trajectory (ps)= "10" #@param ["10", "100", "200", "500", "1000"]
write_the_trajectory_eq = Write_the_trajectory


Write_the_log (ps)= "10" #@param ["10", "100", "200", "500", "1000"]
write_the_log_eq = Write_the_log





In [ ]:
#@title **Run NPT equilibration molecular dynamics simulation**
#@markdown This step ensures that the system reaches a stable thermodynamic state before proceeding to production molecular dynamics simulations.

###########################################
import openmm as mm
from openmm import *
from openmm.app import *
from openmm.unit import *
import pytraj as pt

from sys import stdout, exit, stderr
import os, math, fnmatch

#############################################
# Defining MD simulation parameters

jobname = os.path.join(workDir, Jobname)
coordinatefile = crd
pdbfile = pdb
topologyfile = top

time_ps = float(Time)*1000
simulation_time = float(time_ps)*picosecond		# in ps
dt = int(dt_eq)*femtosecond
temperature = float(temperature_eq)*kelvin
savcrd_freq = int(write_the_trajectory_eq)*picosecond
print_freq  = int(write_the_log_eq)*picosecond

pressure	= float(pressure_eq)*bar

restraint_fc = int(Force_constant) # kJ/mol

nsteps  = int(simulation_time.value_in_unit(picosecond)/dt.value_in_unit(picosecond))
nprint  = int(print_freq.value_in_unit(picosecond)/dt.value_in_unit(picosecond))
nsavcrd = int(savcrd_freq.value_in_unit(picosecond)/dt.value_in_unit(picosecond))

#############################################
# Defining functions to use below:
def backup_old_log(pattern, string):
	result = []
	for root, dirs, files in os.walk("./"):
		for name in files:
			if fnmatch.fnmatch(name, pattern):

				try:
					number = int(name[-2])
					avail = isinstance(number, int)
					#print(name,avail)
					if avail == True:
						result.append(number)
				except:
					pass

	if len(result) > 0:
		maxnumber = max(result)
	else:
		maxnumber = 0

	backup_file = "\#" + string + "." + str(maxnumber + 1) + "#"
	os.system("mv " + string + " " + backup_file)
	return backup_file

def restraints(system, crd, fc, restraint_array):

	boxlx = system.getDefaultPeriodicBoxVectors()[0][0].value_in_unit(nanometers)
	boxly = system.getDefaultPeriodicBoxVectors()[1][1].value_in_unit(nanometers)
	boxlz = system.getDefaultPeriodicBoxVectors()[2][2].value_in_unit(nanometers)

	if fc > 0:
		# positional restraints for all heavy-atoms
		posresPROT = CustomExternalForce('k*periodicdistance(x, y, z, x0, y0, z0)^2;')
		posresPROT.addPerParticleParameter('k')
		posresPROT.addPerParticleParameter('x0')
		posresPROT.addPerParticleParameter('y0')
		posresPROT.addPerParticleParameter('z0')

		for atom1 in restraint_array:
			atom1 = int(atom1)

			xpos  = crd.positions[atom1].value_in_unit(nanometers)[0]
			ypos  = crd.positions[atom1].value_in_unit(nanometers)[1]
			zpos  = crd.positions[atom1].value_in_unit(nanometers)[2]

			posresPROT.addParticle(atom1, [fc, xpos, ypos, zpos])

		system.addForce(posresPROT)

	return system
##############################################

#############################################
print("\n> Simulation details:\n")
print("\tJob name = " + jobname)
print("\tCoordinate file = " + str(coordinatefile))
print("\tPDB file = " + str(pdbfile))
print("\tTopology file = " + str(topologyfile))

print("\n\tSimulation_time = " + str(simulation_time))
print("\tIntegration timestep = " + str(dt))
print("\tTotal number of steps = " +  str(nsteps))

print("\n\tSave coordinates each " + str(savcrd_freq))
print("\tPrint in log file each " + str(print_freq))

print("\n\tTemperature = " + str(temperature))
print("\tPressure = " + str(pressure))
#############################################

print("\n> Setting the system:\n")

if Ligand_Force_field == "OpenFF 2.0.0 (Sage)":
  print("\t- Reading topology and structure file...")
  prmtop = pmd.load_file(topologyfile)
  inpcrd = AmberInpcrdFile(coordinatefile)

  print("\t- Creating system and setting parameters...")
  nonbondedMethod = PME
  nonbondedCutoff = 1.0*nanometers
  ewaldErrorTolerance = 0.0005
  constraints = HBonds
  rigidWater = True
  constraintTolerance = 0.000001
  friction = 1.0
  system = complex_structure.createSystem(nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
                                          constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance)
else:
  print("\t- Reading topology and structure file...")
  prmtop = AmberPrmtopFile(topologyfile)
  inpcrd = AmberInpcrdFile(coordinatefile)

  print("\t- Creating system and setting parameters...")
  nonbondedMethod = PME
  nonbondedCutoff = 1.0*nanometers
  ewaldErrorTolerance = 0.0005
  constraints = HBonds
  rigidWater = True
  constraintTolerance = 0.000001
  friction = 1.0
  system = prmtop.createSystem(nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
                                          constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance)


print("\t- Applying restraints. Force Constant = " + str(Force_constant) + "kJ/mol")
pt_system = pt.iterload(coordinatefile, topologyfile)
pt_topology = pt_system.top
restraint_array = pt.select_atoms('!(:H*) & !(:WAT) & !(:Na+) & !(:Cl-) & !(:Mg+) & !(:K+)', pt_topology)

system = restraints(system, inpcrd, restraint_fc, restraint_array)

print("\t- Setting barostat...")
system.addForce(MonteCarloBarostat(pressure, temperature))

print("\t- Setting integrator...")
integrator = LangevinIntegrator(temperature, friction, dt)
integrator.setConstraintTolerance(constraintTolerance)
simulation = Simulation(prmtop.topology, system, integrator)
simulation.context.setPositions(inpcrd.positions)
if inpcrd.boxVectors is not None:
    simulation.context.setPeriodicBoxVectors(*inpcrd.boxVectors)

print("\t- Energy minimization: " + str(Minimization_steps) + " steps")
simulation.minimizeEnergy(tolerance=10*kilojoule/mole/nanometer, maxIterations=int(Minimization_steps))

print("\t-> Potential Energy = " + str(simulation.context.getState(getEnergy=True).getPotentialEnergy()))

print("\t- Setting initial velocities...")
simulation.context.setVelocitiesToTemperature(temperature)

#############################################
# Running Equilibration on NPT ensemble

dcd_file = jobname + ".dcd"
log_file = jobname + ".log"
rst_file = jobname + ".rst"
prv_rst_file = jobname + ".rst"
pdb_file = jobname + ".pdb"

# Creating a trajectory file and reporters
dcd = DCDReporter(dcd_file, nsavcrd)
firstdcdstep = (nsteps) + nsavcrd
dcd._dcd = DCDFile(dcd._out, simulation.topology, simulation.integrator.getStepSize(), firstdcdstep, nsavcrd) # charmm doesn't like first step to be 0

simulation.reporters.append(dcd)
simulation.reporters.append(StateDataReporter(stdout, nprint, step=True, speed=True, progress=True, totalSteps=nsteps, remainingTime=True, separator='\t\t'))
simulation.reporters.append(StateDataReporter(log_file, nprint, step=True, kineticEnergy=True, potentialEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True))

print("\n> Simulating " + str(nsteps) + " steps...")
simulation.step(nsteps)

simulation.reporters.clear() # remove all reporters so the next iteration don't trigger them.


##################################
# Writing last frame information of stride
print("\n> Writing state file (" + str(rst_file) + ")...")
state = simulation.context.getState( getPositions=True, getVelocities=True )
with open(rst_file, 'w') as f:
	f.write(XmlSerializer.serialize(state))

last_frame = int(nsteps/nsavcrd)
print("> Writing coordinate file (" + str(pdb_file) + ", frame = " + str(last_frame) + ")...")
positions = simulation.context.getState(getPositions=True).getPositions()
PDBFile.writeFile(simulation.topology, positions, open(pdb_file, 'w'))

print("\n> Finished!\n")

---
## **7. Production Molecular Dynamics Simulation**

The production MD simulation is performed using the equilibrated system as the initial configuration. The final state file (.rst), containing atomic positions and velocities, is used to ensure continuity from the equilibration phase.

To accommodate the runtime limitations of the Google Colab environment, the simulation is executed in multiple segments (strides). The total simulation time is defined as:

Total simulation time = Number of strides × Stride time

All generated trajectory segments are concatenated at the end of the simulation to produce a complete trajectory for downstream analysis.

In [ ]:
#@markdown ### **Define input files name for production MD:**

Equilibrated_PDB = 'prot_lig_equil.pdb' #@param {type:"string"}
State_file = 'prot_lig_equil.rst' #@param {type:"string"}
Ligand_Force_field = "GAFF2"
#@markdown ---
#@markdown ### **Parameters for MD Production:**


# remove whitespaces
Jobname = 'prot_lig_prod' #@param {type:"string"}

# Ligand_Force_field = "GAFF2" #@param ["GAFF2", "OpenFF 2.0.0 (Sage)"]

# if Ligand_Force_field == "OpenFF 2.0.0 (Sage)":
#   top = os.path.join(workDir, "SYS_openff.prmtop")
#   crd = os.path.join(workDir, "SYS_openff.inpcrd")
#   pdb = os.path.join(workDir, "SYS.pdb")

# else:
top = os.path.join(workDir, "SYS_gaff2.prmtop")
crd = os.path.join(workDir, "SYS_gaff2.crd")
pdb = os.path.join(workDir, "SYS.pdb")



Stride_Time (ns)= "10" #@param {type:"string"}
stride_time_prod = Stride_Time
Number_of_strides = "1" #@param {type:"string"}
nstride = Number_of_strides
Integration_timestep (fs)= "2" #@param ["0.5", "1", "2", "3", "4"]
dt_prod = Integration_timestep


Temperature (K)= 298 #@param {type:"string"}
temperature_prod = Temperature
Pressure (bar)= 1 #@param {type:"string"}
pressure_prod = Pressure


Write_the_trajectory (ps)= "10" #@param ["10", "100", "200", "500", "1000"]
write_the_trajectory_prod = Write_the_trajectory


Write_the_log (ps) = "10" #@param ["10", "100", "200", "500", "1000"]
write_the_log_prod = Write_the_log

